In [30]:
import json
import re
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

In [17]:
with open("deepseek_results_cleaned.json", "r", encoding="utf-8") as f:
    data = json.load(f)

In [18]:
rows = []
for item in data:
    resp = item.get("response", {})
    rd = next(iter(resp.values()), {}) if isinstance(resp, dict) else {}
    if not isinstance(rd, dict):
        continue
    rows.append({
        "категория":      rd.get("категория", ""),
        "тип_запроса":    rd.get("тип_запроса", ""),
        "кто_выигрывает": rd.get("кто_выигрывает", ""),
        "есть_ли_pg":     rd.get("есть_ли_pg", ""),
        "конкуренты":     rd.get("конкуренты", ""),
    })

df = pd.DataFrame(rows)
cat_order  = ["baby care", "hair care", "oral care"]
type_order = ["брендовый", "категорийный", "консультационный", "сравнительный"]
COLORS = {"P&G": "#00C2B2", "конкуренты": "#FF6B6B", "одинаково": "#AAAAAA"}

In [19]:
pivot = df.groupby(["категория", "тип_запроса", "кто_выигрывает"]).size().unstack(fill_value=0)
pivot = pivot.reindex(columns=["P&G", "конкуренты", "одинаково"], fill_value=0)
pivot["всего"] = pivot.sum(axis=1)
pivot = pivot[pivot["всего"] > 0].reset_index()
pivot["label"] = pivot["категория"].str.replace(" care", "") + "\n" + pivot["тип_запроса"]

fig, ax = plt.subplots(figsize=(14, 6))
bottoms = np.zeros(len(pivot))
for outcome in ["P&G", "конкуренты", "одинаково"]:
    vals = (pivot[outcome] / pivot["всего"] * 100).values
    bars = ax.bar(pivot["label"], vals, bottom=bottoms, color=COLORS[outcome], label=outcome)
    for bar, val, bot in zip(bars, vals, bottoms):
        if val > 5:
            ax.text(bar.get_x() + bar.get_width()/2, bot + val/2,
                    f"{val:.0f}%", ha="center", va="center", fontsize=9, color="white", fontweight="bold")
    bottoms += vals

ax.set_ylim(0, 108)
ax.set_ylabel("Доля запросов, %")
ax.set_xlabel("Категория / Тип запроса")
ax.set_title("P&G vs конкуренты по 12 разрезам\nДоля запросов по исходу выдачи (n=236)", fontsize=13, fontweight="bold")
ax.legend(loc="upper right")
plt.xticks(rotation=15, ha="right")
plt.tight_layout()
plt.savefig("chart1_win_loss_12.png", dpi=150)
plt.close()

In [20]:
wr = df.groupby(["категория", "тип_запроса"]).apply(
    lambda g: round((g["кто_выигрывает"] == "P&G").sum() / len(g) * 100, 1),
    include_groups=False
).unstack()
wr = wr.reindex(index=cat_order, columns=type_order).fillna(0)
wr.index = [i.replace(" care", "") for i in wr.index]

fig, ax = plt.subplots(figsize=(9, 4))
im = ax.imshow(wr.values, cmap="RdYlGn", vmin=0, vmax=100, aspect="auto")
plt.colorbar(im, ax=ax, label="Win Rate %")
ax.set_xticks(range(len(type_order))); ax.set_xticklabels(type_order)
ax.set_yticks(range(len(wr.index))); ax.set_yticklabels(wr.index)
for i in range(len(wr.index)):
    for j in range(len(type_order)):
        val = wr.values[i, j]
        ax.text(j, i, f"{val}%", ha="center", va="center",
                fontsize=13, fontweight="bold",
                color="white" if val > 70 or val < 30 else "black")
ax.set_title("Win Rate P&G — тепловая карта\nзелёный = доминирует, красный = проигрывает", fontsize=12, fontweight="bold")
ax.set_xlabel("Тип запроса"); ax.set_ylabel("Категория")
plt.tight_layout()
plt.savefig("chart2_heatmap_winrate.png", dpi=150)
plt.close()

In [21]:
pr = df.groupby(["категория", "тип_запроса"]).apply(
    lambda g: round((g["есть_ли_pg"] == "Да").sum() / len(g) * 100, 1),
    include_groups=False
).unstack()
pr = pr.reindex(index=cat_order, columns=type_order).fillna(0)
pr.index = [i.replace(" care", "") for i in pr.index]

fig, ax = plt.subplots(figsize=(9, 4))
im = ax.imshow(pr.values, cmap="Blues", vmin=0, vmax=100, aspect="auto")
plt.colorbar(im, ax=ax, label="Presence %")
ax.set_xticks(range(len(type_order))); ax.set_xticklabels(type_order)
ax.set_yticks(range(len(pr.index))); ax.set_yticklabels(pr.index)
for i in range(len(pr.index)):
    for j in range(len(type_order)):
        val = pr.values[i, j]
        ax.text(j, i, f"{val}%", ha="center", va="center",
                fontsize=13, fontweight="bold",
                color="white" if val > 60 else "black")
ax.set_title("Видимость P&G в выдаче — Presence Rate\n% запросов, где P&G упомянут хоть раз", fontsize=12, fontweight="bold")
ax.set_xlabel("Тип запроса"); ax.set_ylabel("Категория")
plt.tight_layout()
plt.savefig("chart3_heatmap_presence.png", dpi=150)
plt.close()

In [22]:
comp_rows = []
for item in data:
    resp = item.get("response", {})
    rd = next(iter(resp.values()), {}) if isinstance(resp, dict) else {}
    if not isinstance(rd, dict): continue
    cat  = rd.get("категория", "")
    conk = rd.get("конкуренты", "")
    if conk:
        for c in str(conk).split(","):
            c = c.strip()
            if c:
                comp_rows.append({"категория": cat, "конкурент": c.lower().title()})

cdf = pd.DataFrame(comp_rows)
top_per_cat = (
    cdf.groupby(["категория", "конкурент"]).size()
    .reset_index(name="count")
    .sort_values(["категория", "count"], ascending=[True, False])
    .groupby("категория").head(5)
)

cat_labels = ["Baby Care", "Hair Care", "Oral Care"]
cat_colors = ["#00C2B2", "#7B61FF", "#FF6B6B"]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, cat, label, color in zip(axes, cat_order, cat_labels, cat_colors):
    sub = top_per_cat[top_per_cat["категория"] == cat].sort_values("count")
    ax.barh(sub["конкурент"], sub["count"], color=color)
    for i, val in enumerate(sub["count"]):
        ax.text(val + 0.2, i, str(val), va="center", fontsize=10)
    ax.set_title(label, fontsize=12, fontweight="bold")
    ax.set_xlabel("Число запросов")
    ax.set_xlim(0, sub["count"].max() * 1.25)

fig.suptitle("Топ конкуренты P&G по категориям", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("chart4_top_competitors.png", dpi=150, bbox_inches="tight")
plt.close()

In [23]:
comp_rows = []
for item in data:
    resp = item.get("response", {})
    rd = next(iter(resp.values()), {}) if isinstance(resp, dict) else {}
    if not isinstance(rd, dict): continue
    cat        = rd.get("категория", "")
    tip        = rd.get("тип_запроса", "")
    conk       = rd.get("конкуренты", "")
    winner     = rd.get("кто_выигрывает", "")
    if conk and winner == "конкуренты":
        for c in str(conk).split(","):
            c = c.strip()
            if c:
                comp_rows.append({"категория": cat, "тип_запроса": tip, "конкурент": c.lower().title()})

cdf = pd.DataFrame(comp_rows)

cat_order  = ["baby care", "hair care", "oral care"]
type_order = ["брендовый", "категорийный", "консультационный", "сравнительный"]
cat_labels = ["Baby Care", "Hair Care", "Oral Care"]
cat_colors = {"baby care": "#00C2B2", "hair care": "#7B61FF", "oral care": "#FF6B6B"}

fig, axes = plt.subplots(3, 4, figsize=(20, 14))
fig.suptitle("Топ конкуренты P&G по категориям и типам запросов",
             fontsize=16, fontweight="bold", y=1.01)

for row_idx, (cat, cat_label) in enumerate(zip(cat_order, cat_labels)):
    for col_idx, tip in enumerate(type_order):
        ax = axes[row_idx][col_idx]

        sub = cdf[(cdf["категория"] == cat) & (cdf["тип_запроса"] == tip)]

        # Заголовок каждого субплота
        ax.set_title(f"{cat_label}\n{tip}", fontsize=9, fontweight="bold")

        if sub.empty:
            ax.text(0.5, 0.5, "нет данных", ha="center", va="center",
                    fontsize=10, color="gray", transform=ax.transAxes)
            ax.axis("off")
            continue

        top5 = (sub.groupby("конкурент").size()
                   .reset_index(name="count")
                   .sort_values("count")
                   .tail(5))

        ax.barh(top5["конкурент"], top5["count"], color=cat_colors[cat])
        for i, val in enumerate(top5["count"]):
            ax.text(val + 0.05, i, str(val), va="center", fontsize=9)
        ax.set_xlim(0, top5["count"].max() * 1.35)
        ax.set_xlabel("Упоминаний", fontsize=8)
        ax.tick_params(axis="y", labelsize=8)
        ax.tick_params(axis="x", labelsize=8)

plt.tight_layout()
plt.savefig("chart4_top_competitors_detailed.png", dpi=150, bbox_inches="tight")
plt.close()
print("Сохранено: chart4_top_competitors_detailed.png")

Сохранено: chart4_top_competitors_detailed.png


In [24]:
with open("deepseek_results_cleaned.json", "r", encoding="utf-8") as f:
    data = json.load(f)

rows = []
for item in data:
    resp = item.get("response", {})
    rd = next(iter(resp.values()), {}) if isinstance(resp, dict) else {}
    if not isinstance(rd, dict): continue
    rows.append({
        "категория":   rd.get("категория", ""),
        "тип_запроса": rd.get("тип_запроса", ""),
    })

df = pd.DataFrame(rows)

# По типу запроса
print("=== По типу запроса ===")
print(df["тип_запроса"].value_counts().to_string())

# По категории
print("\n=== По категории ===")
print(df["категория"].value_counts().to_string())

# Сводная таблица категория × тип
print("\n=== Категория × Тип (кол-во) ===")
print(df.groupby(["категория", "тип_запроса"]).size().unstack(fill_value=0).to_string())


=== По типу запроса ===
тип_запроса
брендовый           216
консультационный    198
категорийный        145
сравнительный        84

=== По категории ===
категория
oral care    233
hair care    209
baby care    201

=== Категория × Тип (кол-во) ===
тип_запроса  брендовый  категорийный  консультационный  сравнительный
категория                                                            
baby care           63            52                59             27
hair care           76            53                70             10
oral care           77            40                69             47


In [25]:
source_rows = []
for item in data:
    resp = item.get("response", {})
    rd = next(iter(resp.values()), {}) if isinstance(resp, dict) else {}
    if not isinstance(rd, dict): continue
    cat = rd.get("категория", "")
    tip = rd.get("тип_запроса", "")
    for src in rd.get("источники", []):
        if not isinstance(src, dict): continue
        source_rows.append({
            "категория":   cat,
            "тип_запроса": tip,
            "домен":       src.get("домен", "").lower().strip(),
            "used":        bool(src.get("used", False)),
            "тип":         src.get("тип", "другое"),
        })

sdf = pd.DataFrame(source_rows)

# ── Агрегация метрик ──
agg = (sdf.groupby(["категория", "тип_запроса", "домен", "тип"])
          .agg(used_count=("used", "sum"), total_count=("used", "count"))
          .reset_index())
agg["usage_rate"] = (agg["used_count"] / agg["total_count"] * 100).round(1)

# ── Сохранение CSV ──
agg.sort_values(["категория","тип_запроса","used_count"], ascending=[True,True,False])\
   .to_csv("sources_full.csv", index=False, encoding="utf-8-sig")

top7 = (agg.sort_values(["категория","тип_запроса","used_count"], ascending=[True,True,False])
           .groupby(["категория","тип_запроса"]).head(7))
top7.to_csv("sources_top7.csv", index=False, encoding="utf-8-sig")

# ── Константы ──
cat_order  = ["baby care", "hair care", "oral care"]
type_order = ["брендовый", "категорийный", "консультационный", "сравнительный"]
cat_labels = {"baby care": "Baby Care", "hair care": "Hair Care", "oral care": "Oral Care"}

TYPE_COLORS = {
    "маркетплейс":       "#00C2B2",
    "перекуп":           "#7B61FF",
    "обзор":             "#FF6B6B",
    "сайт бренда":       "#F5A623",
    "форум":             "#4A90D9",
    "экспертный ресурс": "#27AE60",
    "другое":            "#AAAAAA",
}

legend_patches = [mpatches.Patch(color=c, label=t) for t, c in TYPE_COLORS.items()]


def make_grid(title, filename, bar_col, xlabel, min_total=1, xlim_mult=1.6, vline=None):
    """Универсальная функция для сетки 3×4 горизонтальных баров."""
    fig, axes = plt.subplots(3, 4, figsize=(26, 17))
    fig.suptitle(title, fontsize=16, fontweight="bold", y=1.005)

    for row_i, cat in enumerate(cat_order):
        for col_i, tip in enumerate(type_order):
            ax = axes[row_i][col_i]
            ax.set_title(f"{cat_labels[cat]}  ·  {tip}", fontsize=9,
                         fontweight="bold", pad=5)

            seg = agg[
                (agg["категория"] == cat) &
                (agg["тип_запроса"] == tip) &
                (agg["total_count"] >= min_total)
            ]

            if seg.empty or seg["used_count"].sum() == 0:
                ax.text(0.5, 0.5, "нет данных", ha="center", va="center",
                        fontsize=10, color="#AAAAAA", transform=ax.transAxes)
                ax.axis("off")
                continue

            top = (seg.sort_values(bar_col, ascending=False)
                      .head(7)
                      .sort_values(bar_col))
            colors = [TYPE_COLORS.get(t, "#AAAAAA") for t in top["тип"]]
            bars = ax.barh(top["домен"], top[bar_col], color=colors, height=0.6)

            for bar, row in zip(bars, top.itertuples()):
                used = int(row.used_count)
                rate = row.usage_rate
                val  = getattr(row, bar_col)
                label = f"{val}%  (n={used})" if bar_col == "usage_rate" else f"{used}  ({rate}%)"
                ax.text(bar.get_width() + bar.get_width()*0.04,
                        bar.get_y() + bar.get_height() / 2,
                        label, va="center", fontsize=7.5, color="#333333")

            max_val = top[bar_col].max()
            ax.set_xlim(0, max_val * xlim_mult)
            if vline:
                ax.axvline(x=vline, color="#CCCCCC", linestyle="--", linewidth=0.8)
            ax.set_xlabel(xlabel, fontsize=7.5)
            ax.tick_params(axis="y", labelsize=8)
            ax.tick_params(axis="x", labelsize=7.5)
            ax.spines[["top", "right"]].set_visible(False)

    fig.legend(handles=legend_patches, title="Тип источника", fontsize=9,
               title_fontsize=10, loc="lower center", ncol=7,
               bbox_to_anchor=(0.5, -0.015))
    plt.tight_layout()
    plt.savefig(filename, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"Сохранён: {filename}")

In [26]:
make_grid(
    title="Рейтинг источников по частоте использования (used_count)\nЯндекс Нейро | топ-7 доменов в каждом сегменте",
    filename="chart_sources_used_count.png",
    bar_col="used_count",
    xlabel="Использований (used=True)",
    xlim_mult=1.65,
)

Сохранён: chart_sources_used_count.png


In [27]:
make_grid(
    title="Надёжность источников — Usage Rate (использован / появился, мин. 2 появления)\nЯндекс Нейро | топ-7 доменов в каждом сегменте",
    filename="chart_sources_usage_rate.png",
    bar_col="usage_rate",
    xlabel="Usage Rate %",
    min_total=2,
    xlim_mult=1.5,
    vline=100,
)

Сохранён: chart_sources_usage_rate.png


In [28]:
used_only   = sdf[sdf["used"] == True]
type_pivot  = (used_only.groupby(["категория","тип_запроса","тип"])
                        .size().unstack(fill_value=0))
type_pct    = type_pivot.div(type_pivot.sum(axis=1), axis=0) * 100
type_cols   = [c for c in TYPE_COLORS if c in type_pct.columns]

fig, axes = plt.subplots(3, 4, figsize=(26, 10))
fig.suptitle("Структура типов используемых источников по сегментам (%)",
             fontsize=16, fontweight="bold", y=1.01)

for row_i, cat in enumerate(cat_order):
    for col_i, tip in enumerate(type_order):
        ax = axes[row_i][col_i]
        ax.set_title(f"{cat_labels[cat]}  ·  {tip}", fontsize=9, fontweight="bold", pad=5)
        try:
            row_data = type_pct.loc[(cat, tip), type_cols]
            n_used   = int(type_pivot.loc[(cat, tip)].sum())
        except KeyError:
            ax.text(0.5, 0.5, "нет данных", ha="center", va="center",
                    fontsize=10, color="#AAAAAA", transform=ax.transAxes)
            ax.axis("off")
            continue

        bot = 0
        for t in type_cols:
            v = row_data.get(t, 0)
            if v > 0:
                ax.barh(0, v, left=bot, color=TYPE_COLORS[t], height=0.55)
                if v > 9:
                    ax.text(bot + v / 2, 0, f"{v:.0f}%", ha="center", va="center",
                            fontsize=8.5, color="white", fontweight="bold")
                bot += v

        ax.text(0.99, -0.22, f"n={n_used}", transform=ax.transAxes,
                ha="right", fontsize=8, color="#666666")
        ax.set_xlim(0, 105)
        ax.set_yticks([])
        ax.set_xlabel("Доля %", fontsize=7.5)
        ax.tick_params(axis="x", labelsize=7.5)
        ax.spines[["top","right","left"]].set_visible(False)

fig.legend(handles=legend_patches, title="Тип источника", fontsize=9,
           title_fontsize=10, loc="lower center", ncol=7,
           bbox_to_anchor=(0.5, -0.04))
plt.tight_layout()
plt.savefig("chart_sources_type_structure.png", dpi=150, bbox_inches="tight")
plt.close()
print("Сохранён: chart_sources_type_structure.png")

Сохранён: chart_sources_type_structure.png


In [31]:
SKIP_LEADERS = {"нет","","нерелевантно","нет лидера","не указан","p&g отсутствует"}

def parse_leaders(raw):
    if not raw or str(raw).strip().lower() in SKIP_LEADERS:
        return []
    return [p.strip() for p in re.split(r"[,;]", str(raw))
            if p.strip() and p.strip().lower() not in SKIP_LEADERS]

gap_rows = []
for item in data:
    resp = item.get("response", {})
    rd = next(iter(resp.values()), {}) if isinstance(resp, dict) else {}
    if not isinstance(rd, dict): continue
    cat = rd.get("категория", "")
    tip = rd.get("тип_запроса", "")
    for gap in rd.get("пробелы", []):
        if not isinstance(gap, dict): continue
        gap_type = gap.get("тип", "")
        leaders  = parse_leaders(gap.get("лидер", ""))
        for leader in leaders:
            gap_rows.append({"категория": cat, "тип_запроса": tip,
                             "gap_тип": gap_type, "лидер": leader})
        if not leaders:
            gap_rows.append({"категория": cat, "тип_запроса": tip,
                             "gap_тип": gap_type, "лидер": None})

gdf = pd.DataFrame(gap_rows)

# ── Нормализация типов гэпов ──
GAP_LABELS = {
    "P&G отсутствует в ответе":                         "Отсутствует",
    "P&G не лидирует в рекомендации":                   "Не лидирует",
    "Конкурент доминирует в консультационном сценарии": "Конкурент (консульт.)",
    "Конкурент доминирует в категорийном сценарии":     "Конкурент (категор.)",
    "Конкурент доминирует в сравнительном сценарии":    "Конкурент (сравнит.)",
}
gdf["gap_label"] = gdf["gap_тип"].map(GAP_LABELS).fillna(gdf["gap_тип"])

cat_order  = ["baby care", "hair care", "oral care"]
type_order = ["брендовый", "категорийный", "консультационный", "сравнительный"]
cat_labels = {"baby care": "Baby Care", "hair care": "Hair Care", "oral care": "Oral Care"}

GAP_COLORS = {
    "Отсутствует":           "#FF6B6B",
    "Не лидирует":           "#F5A623",
    "Конкурент (консульт.)": "#7B61FF",
    "Конкурент (категор.)":  "#4A90D9",
    "Конкурент (сравнит.)":  "#AAAAAA",
}
gap_cols = list(GAP_COLORS.keys())

In [33]:
# CSV: полная таблица лидеров и сводка гэпов

leaders_agg = (gdf[gdf["лидер"].notna()]
               .groupby(["категория","тип_запроса","gap_label","лидер"])
               .size().reset_index(name="упоминаний")
               .sort_values(["категория","тип_запроса","gap_label","упоминаний"],
                            ascending=[True,True,True,False]))
leaders_agg.to_csv("gap_leaders_full.csv", index=False, encoding="utf-8-sig")

gap_counts = (gdf.groupby(["категория","тип_запроса","gap_label"])
                 .size().unstack(fill_value=0).reset_index())
gap_counts.to_csv("gap_counts_by_segment.csv", index=False, encoding="utf-8-sig")

In [34]:
# топ-7 конкурентов-лидеров по сегментам

fig, axes = plt.subplots(3, 4, figsize=(26, 17))
fig.suptitle("Конкуренты-лидеры в пробелах P&G по сегментам\n(топ-7 по числу упоминаний в gap.лидер)",
             fontsize=16, fontweight="bold", y=1.005)

for row_i, cat in enumerate(cat_order):
    for col_i, tip in enumerate(type_order):
        ax = axes[row_i][col_i]
        ax.set_title(f"{cat_labels[cat]}  ·  {tip}", fontsize=9, fontweight="bold", pad=5)

        seg = gdf[(gdf["категория"]==cat) & (gdf["тип_запроса"]==tip) & gdf["лидер"].notna()]
        if seg.empty:
            ax.text(0.5,0.5,"нет данных",ha="center",va="center",
                    fontsize=10,color="#AAAAAA",transform=ax.transAxes)
            ax.axis("off")
            continue

        top = (seg.groupby("лидер").size().reset_index(name="count")
                  .sort_values("count").tail(7))

        def dominant_gap(leader):
            sub = seg[seg["лидер"]==leader]["gap_label"]
            return sub.mode()[0] if not sub.empty else "Отсутствует"

        colors = [GAP_COLORS.get(dominant_gap(l), "#AAAAAA") for l in top["лидер"]]
        bars   = ax.barh(top["лидер"], top["count"], color=colors, height=0.6)

        for bar, val in zip(bars, top["count"]):
            ax.text(bar.get_width() + 0.1, bar.get_y() + bar.get_height()/2,
                    str(int(val)), va="center", fontsize=8.5,
                    color="#333333", fontweight="bold")

        ax.set_xlim(0, top["count"].max() * 1.4)
        ax.set_xlabel("Упоминаний в гэпах", fontsize=7.5)
        ax.tick_params(axis="y", labelsize=8)
        ax.tick_params(axis="x", labelsize=7.5)
        ax.spines[["top","right"]].set_visible(False)

legend_patches = [mpatches.Patch(color=c, label=l) for l, c in GAP_COLORS.items()]
fig.legend(handles=legend_patches, title="Преобладающий тип гэпа", fontsize=9,
           title_fontsize=10, loc="lower center", ncol=5, bbox_to_anchor=(0.5, -0.015))
plt.tight_layout()
plt.savefig("chart_gap_leaders_grid.png", dpi=150, bbox_inches="tight")
plt.close()

In [35]:
# структура типов гэпов по сегментам (stacked bar)

gap_type_pivot = (gdf.groupby(["категория","тип_запроса","gap_label"])
                     .size().unstack(fill_value=0))
gap_type_pct   = gap_type_pivot.div(gap_type_pivot.sum(axis=1), axis=0) * 100
present_cols   = [c for c in gap_cols if c in gap_type_pct.columns]

fig, axes = plt.subplots(3, 4, figsize=(26, 11))
fig.suptitle("Структура типов пробелов P&G по сегментам (%)",
             fontsize=16, fontweight="bold", y=1.01)

for row_i, cat in enumerate(cat_order):
    for col_i, tip in enumerate(type_order):
        ax = axes[row_i][col_i]
        ax.set_title(f"{cat_labels[cat]}  ·  {tip}", fontsize=9, fontweight="bold", pad=5)
        try:
            row_data = gap_type_pct.loc[(cat, tip), present_cols]
            n_gaps   = int(gap_type_pivot.loc[(cat, tip)].sum())
        except KeyError:
            ax.text(0.5,0.5,"нет данных",ha="center",va="center",
                    fontsize=10,color="#AAAAAA",transform=ax.transAxes)
            ax.axis("off")
            continue

        bot = 0
        for g in present_cols:
            v = row_data.get(g, 0)
            if v > 0:
                ax.barh(0, v, left=bot, color=GAP_COLORS[g], height=0.55)
                if v > 9:
                    ax.text(bot + v/2, 0, f"{v:.0f}%", ha="center", va="center",
                            fontsize=9, color="white", fontweight="bold")
                bot += v

        ax.text(0.99, -0.25, f"n={n_gaps}", transform=ax.transAxes,
                ha="right", fontsize=8, color="#666666")
        ax.set_xlim(0, 105)
        ax.set_yticks([])
        ax.set_xlabel("Доля %", fontsize=7.5)
        ax.tick_params(axis="x", labelsize=7.5)
        ax.spines[["top","right","left"]].set_visible(False)

legend_patches = [mpatches.Patch(color=GAP_COLORS[g], label=g) for g in present_cols]
fig.legend(handles=legend_patches, title="Тип гэпа", fontsize=9,
           title_fontsize=10, loc="lower center", ncol=5, bbox_to_anchor=(0.5, -0.04))
plt.tight_layout()
plt.savefig("chart_gap_type_structure.png", dpi=150, bbox_inches="tight")
plt.close()


In [36]:
# общий топ-15 конкурентов по всем гэпам (stacked bar)

top_overall = (gdf[gdf["лидер"].notna()]
               .groupby(["лидер","gap_label"]).size().reset_index(name="count"))
top15_names = (top_overall.groupby("лидер")["count"].sum()
                          .nlargest(15).index.tolist())
top_overall = top_overall[top_overall["лидер"].isin(top15_names)]

pivot_c = top_overall.pivot_table(index="лидер", columns="gap_label",
                                  values="count", fill_value=0)
pivot_c["total"] = pivot_c.sum(axis=1)
pivot_c = pivot_c.sort_values("total").drop(columns="total")

fig, ax = plt.subplots(figsize=(13, 8))
bot = pd.Series(0.0, index=pivot_c.index)
for gap in present_cols:
    if gap not in pivot_c.columns: continue
    vals = pivot_c[gap]
    ax.barh(pivot_c.index, vals, left=bot, color=GAP_COLORS[gap], label=gap, height=0.6)
    for y, (v, b) in enumerate(zip(vals, bot)):
        if v > 0:
            ax.text(b + v/2, y, str(int(v)), ha="center", va="center",
                    fontsize=8, color="white", fontweight="bold")
    bot += vals

totals = pivot_c.sum(axis=1)
for y, total in enumerate(totals):
    ax.text(total + 0.2, y, str(int(total)), va="center",
            fontsize=9, color="#333333", fontweight="bold")

ax.set_xlabel("Упоминаний в гэпах", fontsize=11)
ax.set_title("Топ-15 конкурентов по всем пробелам P&G\n(с разбивкой по типу гэпа)",
             fontsize=14, fontweight="bold")
ax.legend(title="Тип гэпа", fontsize=9, title_fontsize=10,
          loc="lower right", framealpha=0.9)
ax.set_xlim(0, totals.max() * 1.2)
ax.spines[["top","right"]].set_visible(False)
plt.tight_layout()
plt.savefig("chart_gap_leaders_overall.png", dpi=150, bbox_inches="tight")
plt.close()